# MIDI ingestion: feature-based deduplication and split planning

Add any new MIDI collection as a direct subfolder of `../data/new_midi_files`. This notebook discovers those source folders automatically, identifies Beethoven, Chopin, and Mozart files from their filenames or enclosing folders, and writes a duplicate/split plan without deleting any MIDI files.


In [1]:
from pathlib import Path
import hashlib

import pandas as pd
import pretty_midi

COMPOSERS = ('Beethoven', 'Chopin', 'Mozart')
PREFERRED_DIRS = {composer: Path('../data/midiclassics') / composer for composer in COMPOSERS}
NEW_DATA_DIR = Path('../data/new_midi_files')
MIDI_SUFFIXES = {'.mid', '.midi'}
SPLIT_PLAN_PATH = Path('../data/split_plans/candidate_split_plan.csv')


In [2]:
def midi_files(directory):
    return sorted(path for path in directory.rglob('*') if path.is_file() and path.suffix.casefold() in MIDI_SUFFIXES)

def source_folders():
    if not NEW_DATA_DIR.is_dir():
        raise FileNotFoundError(f'Create the new-data folder first: {NEW_DATA_DIR.resolve()}')
    return sorted((path for path in NEW_DATA_DIR.iterdir() if path.is_dir()), key=lambda path: path.name.casefold())

def composer_from_path(path):
    parts = [path.stem, *path.relative_to(NEW_DATA_DIR).parts[:-1]]
    normalized_parts = [part.casefold() for part in parts]
    for composer in COMPOSERS:
        name = composer.casefold()
        if any(name == part or name in part for part in normalized_parts):
            return composer
    return None

def feature_record(path, source, composer):
    midi = pretty_midi.PrettyMIDI(str(path))
    duration = midi.get_end_time()
    num_notes = sum(len(instrument.notes) for instrument in midi.instruments)
    if duration <= 0 or num_notes == 0:
        raise ValueError('MIDI has no timed notes')
    tempo = midi.estimate_tempo()
    notes_per_second = num_notes / duration
    signature = (round(duration, 6), round(tempo, 6), num_notes, round(notes_per_second, 9))
    return {'source': source, 'composer': composer, 'path': path, 'total_duration': duration,
            'estimated_tempo': tempo, 'num_notes': num_notes, 'notes_per_second': notes_per_second,
            'feature_signature': signature}

def collect_records(directory, source, fixed_composer=None):
    records, errors, ignored = [], [], []
    for path in midi_files(directory):
        composer = fixed_composer or composer_from_path(path)
        if composer not in COMPOSERS:
            ignored.append({'source': source, 'path': str(path), 'reason': 'composer_not_identified'})
            continue
        try:
            records.append(feature_record(path, source, composer))
        except Exception as error:
            errors.append({'source': source, 'composer': composer, 'path': str(path), 'error': str(error)})
    return records, errors, ignored

def assigned_split(composer, signature):
    value = int(hashlib.sha256(repr((composer, signature)).encode()).hexdigest()[:8], 16) % 100
    return 'train' if value < 70 else 'dev' if value < 85 else 'test'


In [3]:
preferred_by_key, errors = {}, []
for composer, directory in PREFERRED_DIRS.items():
    records, source_errors, _ = collect_records(directory, 'midiclassics', composer)
    errors.extend(source_errors)
    for record in records:
        preferred_by_key.setdefault((composer, record['feature_signature']), []).append(record['path'])

candidate_by_key, ignored = {}, []
for source_directory in source_folders():
    source = source_directory.name
    records, source_errors, source_ignored = collect_records(source_directory, source)
    errors.extend(source_errors)
    ignored.extend(source_ignored)
    for record in records:
        candidate_by_key.setdefault((record['composer'], record['feature_signature']), []).append(record)

priority = {source: rank for rank, source in enumerate(sorted({record['source'] for records in candidate_by_key.values() for record in records}, key=str.casefold))}
plan = []
for (composer, signature), records in candidate_by_key.items():
    records.sort(key=lambda record: (priority[record['source']], str(record['path']).casefold()))
    preferred_paths = preferred_by_key.get((composer, signature), [])
    keeper = None if preferred_paths else records[0]
    for record in records:
        if preferred_paths:
            status, duplicate_of, split = 'duplicate_of_midiclassics', ' | '.join(map(str, preferred_paths)), None
        elif record is keeper:
            status, duplicate_of, split = ('keep_unique' if len(records) == 1 else 'keep_candidate_representative'), '', assigned_split(composer, signature)
        else:
            status, duplicate_of, split = 'duplicate_of_candidate', str(keeper['path']), None
        plan.append({'source': record['source'], 'composer': composer, 'path': str(record['path']),
                     'status': status, 'duplicate_of': duplicate_of, 'recommended_split': split,
                     'total_duration': record['total_duration'], 'estimated_tempo': record['estimated_tempo'],
                     'num_notes': record['num_notes'], 'notes_per_second': record['notes_per_second']})

report = pd.DataFrame(plan).sort_values(['composer', 'status', 'source', 'path'], ignore_index=True)
error_report = pd.DataFrame(errors)
ignored_report = pd.DataFrame(ignored)
SPLIT_PLAN_PATH.parent.mkdir(parents=True, exist_ok=True)
report.to_csv(SPLIT_PLAN_PATH, index=False)
display(report)
display(report.groupby(['composer', 'status']).size().rename('files').to_frame())
display(error_report)
display(ignored_report)
print(f'Scanned {len(source_folders()):,} source folders; saved {len(report):,} planned files to {SPLIT_PLAN_PATH}. No MIDI files were deleted.')


c:\Users\Maxtw\OneDrive\Desktop\511 Project\.venv\Lib\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


,source,composer,path,status,duplicate_of,recommended_split,total_duration,estimated_tempo,num_notes,notes_per_second
0,BigSynthPiano,Beethoven,..\data\new_midi_files\BigSynthPiano\midi\Beet...,duplicate_of_candidate,..\data\new_midi_files\BigSynthPiano\midi\Beet...,NaN,15.994452,179.999910,255,15.943028
1,BigSynthPiano,Beethoven,..\data\new_midi_files\BigSynthPiano\midi\Beet...,duplicate_of_candidate,..\data\new_midi_files\BigSynthPiano\midi\Beet...,NaN,22.140386,82.105256,66,2.980978
2,BigSynthPiano,Beethoven,..\data\new_midi_files\BigSynthPiano\midi\Beet...,duplicate_of_candidate,..\data\new_midi_files\BigSynthPiano\midi\Beet...,NaN,14.398750,200.000000,172,11.945481
3,BigSynthPiano,Beethoven,..\data\new_midi_files\BigSynthPiano\midi\Beet...,duplicate_of_candidate,..\data\new_midi_files\BigSynthPiano\midi\Beet...,NaN,13.082965,219.999817,131,10.013020
4,BigSynthPiano,Beethoven,..\data\new_midi_files\BigSynthPiano\midi\Beet...,duplicate_of_candidate,..\data\new_midi_files\BigSynthPiano\midi\Beet...,NaN,13.076147,219.999817,224,17.130428
...,...,...,...,...,...,...,...,...,...,...
1324,midisheetmusic.com,Mozart,..\data\new_midi_files\midisheetmusic.com\Moza...,keep_unique,,dev,97.997917,216.450939,828,8.449159
1325,midisheetmusic.com,Mozart,..\data\new_midi_files\midisheetmusic.com\Moza...,keep_unique,,train,214.393333,187.226277,1009,4.706303
1326,midisheetmusic.com,Mozart,..\data\new_midi_files\midisheetmusic.com\Moza...,keep_unique,,test,100.796667,130.434783,660,6.547836
1327,midisheetmusic.com,Mozart,..\data\new_midi_files\midisheetmusic.com\Moza...,keep_unique,,train,153.597500,196.629213,1608,10.468920


files
composer  status                              
Beethoven duplicate_of_candidate             7
          duplicate_of_midiclassics        220
          keep_candidate_representative      7
          keep_unique                      290
Chopin    duplicate_of_candidate            40
          duplicate_of_midiclassics        155
          keep_candidate_representative     40
          keep_unique                      145
Mozart    duplicate_of_candidate            39
          duplicate_of_midiclassics        257
          keep_candidate_representative     39
          keep_unique                       90

,source,composer,path,error
0,midiclassics,Beethoven,..\data\midiclassics\Beethoven\Anhang 14-3.mid,Could not decode key with 3 flats and mode 255
1,midiclassics,Mozart,..\data\midiclassics\Mozart\Piano Sonatas\Nuev...,Could not decode key with 2 flats and mode 2
2,commons.wikimedia.org,Beethoven,..\data\new_midi_files\commons.wikimedia.org\B...,0 is not a valid `numerator` type or value
3,commons.wikimedia.org,Beethoven,..\data\new_midi_files\commons.wikimedia.org\B...,0 is not a valid `numerator` type or value
4,commons.wikimedia.org,Beethoven,..\data\new_midi_files\commons.wikimedia.org\B...,Could not decode key with 8 sharps and mode 0
5,commons.wikimedia.org,Chopin,..\data\new_midi_files\commons.wikimedia.org\C...,Could not decode key with 8 flats and mode 1
6,midi-classical-music,Beethoven,..\data\new_midi_files\midi-classical-music\da...,Could not decode key with 3 flats and mode 255
7,midi-classical-music,Mozart,..\data\new_midi_files\midi-classical-music\da...,Could not decode key with 2 flats and mode 2


""


Scanned 8 source folders; saved 1,329 planned files to ..\data\split_plans\candidate_split_plan.csv. No MIDI files were deleted.
